# 1. Import the Data

In [38]:
import pandas as pd

df = pd.read_csv("data.csv")
df.head()

,title,score,comments
0,GameStop makes $55.5B takeover offer for eBay,187,122
1,ASML's Best Selling Product Isn't What You Thi...,45,22
2,Trademark violation: Fake Notepad++ for Mac,234,98
3,Using “underdrawings” for accurate text and nu...,271,92
4,Texico: Learn the principles of programming wi...,75,4


In [39]:
# Clean the text
df["title"] = df["title"].str.lower()

# Create labels
threshold = df["score"].quantile(0.7)           # We will consider the top 30% scored posts as positive, instead of using a fixed arbitrary threshold
df["label"] = (df["score"] > threshold).astype(int)

# Selecting only what we need
df = df[["title", "label"]]
df.head()

,title,label
0,gamestop makes $55.5b takeover offer for ebay,0
1,asml's best selling product isn't what you thi...,0
2,trademark violation: fake notepad++ for mac,0
3,using “underdrawings” for accurate text and nu...,1
4,texico: learn the principles of programming wi...,0


In [40]:
# Save new csv

df.to_csv("clean_data.csv", index=False)

# 2. Text Tokenization and Numerical Encoding

In [41]:
import torch
from collections import Counter

# Tokenization + Encoding

# Build vocabulary

all_words = " ".join(df["title"]).split()
word_counts = Counter(all_words)

vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.most_common(5000))}


# Encode text

def encode(text):
    return [vocab.get(word, 0) for word in text.split()]

df["encoded"] = df["title"].apply(encode)


# Pad sequences

max_len = 10

def pad(seq):
    return seq[:max_len] + [0]*(max_len - len(seq))

df["padded"] = df["encoded"].apply(pad)

# Check
df.head()


,title,label,encoded,padded
0,gamestop makes $55.5b takeover offer for ebay,0,"[44, 45, 46, 47, 48, 5, 49]","[44, 45, 46, 47, 48, 5, 49, 0, 0, 0]"
1,asml's best selling product isn't what you thi...,0,"[50, 51, 52, 53, 54, 17, 55, 56, 13, 6]","[50, 51, 52, 53, 54, 17, 55, 56, 13, 6]"
2,trademark violation: fake notepad++ for mac,0,"[57, 58, 59, 60, 5, 61]","[57, 58, 59, 60, 5, 61, 0, 0, 0, 0]"
3,using “underdrawings” for accurate text and nu...,1,"[62, 63, 5, 64, 18, 4, 65]","[62, 63, 5, 64, 18, 4, 65, 0, 0, 0]"
4,texico: learn the principles of programming wi...,0,"[66, 67, 1, 68, 3, 69, 70, 71, 72, 2, 19]","[66, 67, 1, 68, 3, 69, 70, 71, 72, 2]"


# 3. The Model

## Model Definition

In [52]:
# Convert to tensors

X = torch.tensor(df["padded"].tolist(), dtype=torch.long)
y = torch.tensor(df["label"].values, dtype=torch.float32)

print(X.shape, y.shape)

torch.Size([50, 10]) torch.Size([50])


In [53]:
df["label"].value_counts()

label
0    35
1    15
Name: count, dtype: int64

In [54]:
# Train test split
from sklearn.model_selection import train_test_split

# Stratify to preserve class distribution across train and test sets for fair evaluation and avoid a "lazy" model.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [72]:
# Define the model
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)     # (batch, seq, embed)
        x = x.mean(dim=1)         # average words
        x = self.fc(x)
        return self.sigmoid(x)

In [73]:
# Initialize the model

vocab_size = len(vocab) + 1
model = SimpleModel(vocab_size, embed_dim=16)

In [74]:
# Training setup

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [75]:
# Training loop
for epoch in range(10):
    model.train()
    
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7536
Epoch 2, Loss: 0.7321
Epoch 3, Loss: 0.7118
Epoch 4, Loss: 0.6926
Epoch 5, Loss: 0.6742
Epoch 6, Loss: 0.6568
Epoch 7, Loss: 0.6400
Epoch 8, Loss: 0.6240
Epoch 9, Loss: 0.6086
Epoch 10, Loss: 0.5937


In [76]:
# Test predictions

model.eval()

with torch.no_grad():
    preds = model(X_test).squeeze()
    
    print("Probabilities:", preds[:10])
    
    threshold = 0.5
    pred_labels = (preds > threshold).float()
    
    accuracy = (pred_labels == y_test).float().mean()
    print("Test Accuracy:", accuracy.item())
    
    print("\nPredictions vs True:")
    for i in range(len(pred_labels)):
        print(f"Prob: {preds[i]:.3f} | Pred: {pred_labels[i].item()} | True: {y_test[i].item()}")

Probabilities: tensor([0.3258, 0.4350, 0.5839, 0.4303, 0.5730, 0.3664, 0.4215, 0.4022, 0.4032,
        0.3635])
Test Accuracy: 0.699999988079071

Predictions vs True:
Prob: 0.326 | Pred: 0.0 | True: 0.0
Prob: 0.435 | Pred: 0.0 | True: 0.0
Prob: 0.584 | Pred: 1.0 | True: 1.0
Prob: 0.430 | Pred: 0.0 | True: 0.0
Prob: 0.573 | Pred: 1.0 | True: 0.0
Prob: 0.366 | Pred: 0.0 | True: 1.0
Prob: 0.421 | Pred: 0.0 | True: 1.0
Prob: 0.402 | Pred: 0.0 | True: 0.0
Prob: 0.403 | Pred: 0.0 | True: 0.0
Prob: 0.363 | Pred: 0.0 | True: 0.0


In [77]:
print(pred_labels)
print(y_test)

tensor([0., 0., 1., 0., 1., 0., 0., 0., 0., 0.])
tensor([0., 0., 1., 0., 0., 1., 1., 0., 0., 0.])


## Conclusion

- The model was evaluated on a held-out test set and achieved approximately 70% accuracy.

- Predicted probabilities are mostly concentrated around the mid-range (≈0.35–0.58), indicating that the model has limited confidence and struggles to clearly separate the two classes.

- Overall, the model demonstrates basic learning but remains a weak classifier, serving as a baseline for further improvements in data quality (small dataset size), class balance, and model architecture.